# 数值精度与结果验证

学习目标：识别浮点误差与非有限值，根据任务选择容差，并分别检查数组形状、类型和数值。

前置知识：浮点类型、数组算术、聚合、布尔数组、异常捕获。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

后续单元沿用首次导入的 np。输入均为本章构造的小数组。

## 1 检查计算结果

计算三个 0.1 的总和，数学结果是 0.3，浮点结果却可能无法用 == 通过检查。大多数十进制小数不能用有限的二进制位精确表示，每次计算还可能产生新的舍入误差。

np.allclose() 检查所有元素是否在容差内接近。本例只演示很少次数的 float64 运算，约定绝对容差为 1e-15、相对容差为 0；这不是所有数值任务通用的误差标准。

In [1]:
import numpy as np

values = np.array([0.1, 0.1, 0.1], dtype=np.float64)
total = np.sum(values)

print(repr(float(total)))  # 0.30000000000000004，显示更多有效位。
print(total == 0.3)  # False，精确比较不通过。
print(np.allclose(total, 0.3, rtol=0, atol=1e-15))  # True，在本例约定容差内。

0.30000000000000004
False
True


## 2 精度、舍入与消减

### 2.1 选择计算类型

finfo().eps 是 1 与该类型中紧邻其上的可表示数之间的差。它描述类型在 1 附近的精度，不能直接当成所有量级和算法的误差上限。

两个相近的大量相减时，原来很小的表示误差可能成为结果中的主要误差，这称为消减（cancellation）问题。下面 1.00000001 在 float32 中已经舍入成 1，之后转为 float64 也无法恢复丢失的差值。

In [2]:
narrow = np.array([1.00000001, 1.0], dtype=np.float32)
wide = np.array([1.00000001, 1.0], dtype=np.float64)

print(np.finfo(np.float32).eps, np.finfo(np.float64).eps)
# 约为 1.19e-7 与 2.22e-16，float64 在 1 附近间隔更小。
print(narrow[0] - narrow[1])  # 0.0，输入存储时已经失去差值。
print(wide[0] - wide[1])  # 约 1e-8，仍不是精确十进制差值。
promoted = narrow.astype(np.float64)
print(promoted[0] - promoted[1])  # 0.0，事后扩宽不能恢复输入信息。

1.1920929e-07 2.220446049250313e-16
0.0
9.99999993922529e-09
0.0


### 2.2 累计过程中的误差

聚合的精度还受计算类型与求和方式影响。NumPy 在许多情形下使用部分成对求和改善精度，但不能保证所有结果都精确，也不保证不同轴和布局得到完全相同的误差。

下面三个输入本身可以由 float32 表示；用 float64 累计能保留这个例子中较小的加数。不要将一次例子的改善推成所有输入都无误差。

In [3]:
values = np.array([1e8, 1.0, -1e8], dtype=np.float32)
narrow_total = np.sum(values, dtype=np.float32)
wide_total = np.sum(values, dtype=np.float64)

print(narrow_total, narrow_total.dtype)  # 0.0 float32，小的贡献在累计中丢失。
print(wide_total, wide_total.dtype)  # 1.0 float64，与本例手算结果一致。
print(np.round(np.array([0.1]), 1) * 3 == np.array([0.3]))
# [False]，预先按小数位舍入不等于改用了精确十进制算术。

0.0 float32
1.0 float64
[False]


## 3 绝对容差与相对容差

### 3.1 比较公式

对于有限数值，isclose(a, b) 按下面条件逐元素比较，allclose() 再要求所有比较都成立：

abs(a - b) ≤ atol + rtol × abs(b)

a 是待检查值，b 是参考值；atol 是与数值同单位的绝对容差，rtol 是无单位的相对容差。接近零时通常需要关注绝对容差，量级较大时相对项可以表达按比例允许的偏差。阈值应来自任务误差要求或数值分析，不能只为了让检查通过而放宽。

下面假定某任务允许绝对误差 0.01，另允许参考值的十万分之一；这只是本例的人为验收约定。

In [4]:
actual = np.array([0.005, 1000.015, 1000.030])
reference = np.array([0.0, 1000.0, 1000.0])
limits = 0.01 + 1e-5 * np.abs(reference)

print(limits)  # [0.01 0.02 0.02]，逐元素允许偏差。
print(np.isclose(actual, reference, rtol=1e-5, atol=0.01))  # [True True False]。
print(np.allclose(actual, reference, rtol=1e-5, atol=0.01))  # False，末项超限。

[0.01 0.02 0.02]
[ True  True False]
False


### 3.2 近零与参考方向

默认 atol=1e-8 可能对很小的目标过于宽松。例如两数都接近零，但相差一倍，默认比较仍可能返回 True。

公式使用第二个输入作为参考，因此一般并不对称。下面用较大的相对容差展示这种规则，实际验收不能照搬这个演示阈值。

In [5]:
print(np.isclose(1e-9, 2e-9))  # True，差值被默认绝对容差覆盖。
print(np.isclose(1e-9, 2e-9, rtol=0.01, atol=0))  # False，改为仅允许 1% 相对偏差。
print(np.isclose(1.0, 2.0, rtol=0.6, atol=0))  # True，参考值为 2。
print(np.isclose(2.0, 1.0, rtol=0.6, atol=0))  # False，参考值变成 1。

True
False
True
False


## 4 非有限值

NaN 表示非数值结果，也常用于缺失标记；正负 Inf 表示正负无穷。isfinite() 仅对有限数值返回 True，isnan() 和 isinf() 分别识别 NaN 与无穷。

allclose() 默认不把 NaN 当作相等；equal_nan=True 只表示相同位置的 NaN 可通过比较，不表示那些位置变成了有效观测。相同位置、同号的无穷可以比较为相等。任务要求有限结果时，还需单独检查。

In [6]:
values = np.array([1.0, np.nan, np.inf, -np.inf])

print(np.isfinite(values))  # [True False False False]。
print(np.isnan(values))  # [False True False False]。
print(np.isinf(values))  # [False False True True]。
print(np.allclose(values, values))  # False，NaN 默认不相等。
print(np.allclose(values, values, equal_nan=True))  # True，不代表所有结果有效。
print(np.all(np.isfinite(values)))  # False，有限性需要另外检查。

[ True False False False]
[False  True False False]
[False False  True  True]
False
True
False


## 5 局部浮点错误策略

errstate() 在 with 块中设置除零、溢出、下溢或无效运算的处理方式，退出时恢复原设置。raise 会把对应情况变为 FloatingPointError；它不能检测所有舍入误差或精度损失。

下面分别检查浮点除零、负实数平方根和指数溢出。这里捕获指定异常以展示边界，实际程序应根据任务选择拒绝输入、调整计算或处理结果。

In [7]:
before = np.geterr()
for name, operation in [
    ("除零", lambda: np.divide(1.0, 0.0)),
    ("无效运算", lambda: np.sqrt(-1.0)),
    ("溢出", lambda: np.exp(1000.0)),
]:
    try:
        with np.errstate(divide="raise", invalid="raise", over="raise"):
            operation()
    except FloatingPointError as error:
        print(name, type(error).__name__)  # 三种情况均显示 FloatingPointError。
    else:
        raise AssertionError("本例应触发指定的浮点异常")
print(np.geterr() == before)  # True，局部设置已经恢复。

除零 FloatingPointError
无效运算 FloatingPointError
溢出 FloatingPointError
True


## 6 形状、类型与数值

### 6.1 接近不等于结构正确

allclose() 支持广播，因此不同形状也可能比较为 True。它也不要求 dtype 相同。array_equal() 则要求形状相同、元素精确相等，但仍不会替代类型检查。

下面一维和二维数组在广播后数值一致，但不能据此认为它们符合相同的输出接口。

In [8]:
vector = np.array([1.0, 2.0], dtype=np.float64)
row = np.array([[1.0, 2.0]], dtype=np.float64)
integers = np.array([1, 2], dtype=np.int64)

print(vector.shape, row.shape)  # (2,) (1, 2)。
print(np.allclose(vector, row))  # True，允许广播。
print(np.array_equal(vector, row))  # False，形状不同。
print(np.array_equal(vector, integers))  # True，元素相等不要求同 dtype。
print(vector.dtype == integers.dtype)  # False，类型单独检查。

(2,) (1, 2)
True
False
True
False


### 6.2 用断言表达要求

numpy.testing 的断言在条件不满足时抛出 AssertionError。assert_allclose() 检查容差；strict=True 还要求形状与类型相同，并禁用标量的特殊广播行为。assert_array_equal() 用于精确元素检查，也可用 strict=True 检查类型。

下面用明确的有限性、结构与容差要求核对两项 float64 结果。assert_allclose() 的默认容差及 equal_nan 默认值与 allclose() 不完全相同，因此显式写出本例条件。

In [9]:
actual = np.array([0.1 + 0.2, 0.5], dtype=np.float64)
reference = np.array([0.3, 0.5], dtype=np.float64)

assert np.all(np.isfinite(actual))
np.testing.assert_allclose(actual, reference, rtol=0, atol=1e-15, equal_nan=False, strict=True)
print("有限性、形状、类型与数值检查通过")

try:
    np.testing.assert_allclose(actual.astype(np.float32), reference, rtol=1e-6, atol=0, strict=True)
except AssertionError:
    print("类型检查拒绝 float32 结果")  # 数值虽接近，类型不满足 float64 要求。
else:
    raise AssertionError("strict=True 应检查 dtype")

np.testing.assert_array_equal(np.array([1, 2]), np.array([1, 2]), strict=True)
print("整数精确检查通过")

有限性、形状、类型与数值检查通过
类型检查拒绝 float32 结果
整数精确检查通过


## 7 选学：稳定计算的专用函数

### 7.1 接近零的指数与对数

log1p(x) 计算 log(1 + x)，expm1(x) 计算 exp(x) - 1；x 很小时，它们能避免直接表达式中加一或减一引入的精度损失。实数 log1p 输入仍要满足相应的定义域条件，本例 x 为正数。

下面 x=1e-16；普通 float64 运算中，1 + x 已舍入为 1。

In [10]:
x = np.float64(1e-16)
print(np.log(1 + x), np.log1p(x))  # 0.0 与约 1e-16。
print(np.exp(x) - 1, np.expm1(x))  # 0.0 与约 1e-16。
print(1 + x == 1)  # True，直接加一已失去小增量。

0.0 1e-16
0.0 1e-16
True


### 7.2 对数域中的加法

logaddexp(a, b) 计算 log(exp(a) + exp(b))，可在直接指数溢出的场景中完成计算。a、b 是两个实数对数值；这里不需要把各自的指数显式存下来。

本例 a=b=1000，数学结果为 1000 + log(2)，数值约为 1000.693147。

In [11]:
result = np.logaddexp(1000.0, 1000.0)
reference = 1000.0 + np.log(2.0)
print(result)  # 约 1000.693147，结果有限。
print(np.isfinite(result))  # True。
print(np.allclose(result, reference, rtol=0, atol=1e-12))  # True，本例少量 float64 运算的约定。

1000.6931471805599
True
True


## 8 选学：seterr 与恢复设置

seterr() 修改后续 NumPy 运算使用的错误策略，不会在当前语句结束后自动恢复。交互环境中，这种设置可能影响后面的实验；需要局部策略时优先用 errstate()。

下面保存旧设置，并在 finally 中恢复。这里只展示当前执行上下文中的持续作用，不讨论多线程与异步上下文的传播。

In [12]:
old_settings = np.seterr(over="raise")
try:
    print(np.geterr()["over"])  # raise，对后续运算持续生效。
finally:
    np.seterr(**old_settings)
print(np.geterr() == old_settings)  # True，恢复进入前的策略。

raise
True


## 本章小结

（1）浮点存储与计算都可能产生误差；扩大累计类型可能改善结果，但不能恢复已经丢失的输入信息。

（2）容差比较以第二个输入为参考；近零、较大数值和非有限值需要明确不同条件。

（3）allclose 可以广播，也不检查 dtype。完整验收应分别考虑形状、类型、有限性与数值。

（4）errstate 处理指定浮点异常，不是所有精度问题的探测器；稳定函数可以针对具体表达式减少精度损失。

## 练习

（1）先预测三种比较结果，再运行。说明哪些检查了结构，哪些只说明广播后的数值接近。

In [13]:
left = np.array([1.0, 2.0])
right = np.array([[1.0, 2.0]])

# 先写下预测，结合两者 shape 解释原因。
print(np.allclose(left, right))
print(np.array_equal(left, right))
print(left.shape == right.shape)

True
False
False


（2）一组近零结果要求绝对误差不超过 1e-10，另一组较大结果只允许参考值的百万分之一偏差。分别选择 atol、rtol 并解释理由，不采用默认容差。

In [14]:
near_actual = np.array([5e-11, 2e-10])
near_reference = np.array([0.0, 0.0])
large_actual = np.array([1000000.5, 1000002.0])
large_reference = np.array([1000000.0, 1000000.0])

# 在此分别调用 isclose 并说明参数选择。
# 检查：两组逐元素结果都应为 [True False]；说明两组阈值的单位。

（3）输出必须是形状 (2,) 的 float64 有限数组，数值与参考值的绝对偏差不超过 1e-12。完成检查，再将输入改为含 NaN 的数组，说明哪个条件应拒绝它。

In [15]:
actual = np.array([0.1 + 0.2, 1.0], dtype=np.float64)
reference = np.array([0.3, 1.0], dtype=np.float64)

# 在此分别检查有限性、结构和数值，明确 equal_nan=False。
# 如果演示拒绝含 NaN 的结果，捕获预期 AssertionError，保持整章可顺序运行。

（4）选学：对很小的正数计算 log(1+x) 与 exp(x)-1。选择适合的函数并说明直接表达式中哪一步可能丢失信息。

In [16]:
x = np.float64(1e-16)

# 在此比较直接表达式和专用函数；检查专用函数的结果有限、为正且接近 1e-16。
# 说明应选的比较阈值，不能用默认 atol 把 0 也验收为正确的小量。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（Python 3.12） | [Floating-Point Arithmetic: Issues and Limitations](https://docs.python.org/3.12/tutorial/floatingpoint.html) 的二进制表示、显示舍入、预先 round 的限制与多次求和／消减示例。 |
| NumPy 官方文档（NumPy 2.5） | [finfo](https://numpy.org/doc/2.5/reference/generated/numpy.finfo.html) 的 eps；[sum](https://numpy.org/doc/2.5/reference/generated/numpy.sum.html) 的 dtype 与 Notes 中的部分成对求和；[isclose](https://numpy.org/doc/2.5/reference/generated/numpy.isclose.html)、[allclose](https://numpy.org/doc/2.5/reference/generated/numpy.allclose.html) 的比较公式、非对称性、默认近零容差、广播与非有限值；[array_equal](https://numpy.org/doc/2.5/reference/generated/numpy.array_equal.html) 的形状和相等条件；[isfinite](https://numpy.org/doc/2.5/reference/generated/numpy.isfinite.html) 的有限性定义；[errstate](https://numpy.org/doc/2.5/reference/generated/numpy.errstate.html)、[seterr](https://numpy.org/doc/2.5/reference/generated/numpy.seterr.html) 的异常类别、raise 与恢复；[assert_allclose](https://numpy.org/doc/2.5/reference/generated/numpy.testing.assert_allclose.html)、[assert_array_equal](https://numpy.org/doc/2.5/reference/generated/numpy.testing.assert_array_equal.html) 的默认条件与 strict；[log1p](https://numpy.org/doc/2.5/reference/generated/numpy.log1p.html)、[expm1](https://numpy.org/doc/2.5/reference/generated/numpy.expm1.html) 的小量精度 Notes；[logaddexp](https://numpy.org/doc/2.5/reference/generated/numpy.logaddexp.html) 的定义与对数域用途。 |